In [26]:
import fitz
import re, os
from pathlib import Path
import random
import time

NUM_LINE_RE = re.compile(
    r"^\s*[\(\[\{]?\s*[₹$€£]?\s*[-+]?"
    r"\d[\d,]*(?:\.\d+)?%?"
    r"\s*[\)\]\}]?\s*$"
)

SECTION_RE = re.compile(
    r"\b(?:financial\s+statements|standalone|consolidated|balance\s+sheet|profit\s+(?:&|and)\s+loss|cash\s+flow)\b",
    re.I
)

In [ ]:
def page_metadata(page, rw, rh):
    
    w = page.rect.width
    h = page.rect.height
    a = round(w*h,2)
    is_segmented = (w/rw) >1.6 and (h/rh) > 0.7
    
    
    blocks = page.get_text("dict")["blocks"]
    
    mar_x = w * 0.05
    mar_y = h * 0.15

    work_rect = fitz.Rect(
        mar_x,
        mar_y,
        w - mar_x,
        h - mar_y
    )
    
    numeric_lines = []
    segment_lines = []
    page_text = []
    dirs = Counter()    

    for block in blocks:

        if block["type"] != 0:
            continue
        for line in block["lines"]:
            bbox = fitz.Rect(line["bbox"])
            if not work_rect.intersects(bbox):
                continue
            
            text = "".join(span["text"] for span in line["spans"]).strip()
            
            if not text:
                continue
            
            dirs[tuple(map(round, line["dir"]))] += 1
            page_text.append(text)
            
            #REGEX
            if NUM_LINE_RE.match(text):
                numeric_lines.append({
                    "text": text,
                    "bbox": bbox,
                    "cx": (bbox.x0 + bbox.x1) / 2
                })
            
            s_match = SECTION_RE.findall(text)
            if s_match:
                segment_lines.append(s_match)

    p_dir = dirs.most_common(1)[0][0] if dirs else None
    text_dir = {
        (0, -1): "90_CCLK",
        (0, 1): "90_CLK",
        (-1, 0): "UPSIDE_DOWN",
        }.get(p_dir, "NORMAL")
    
    return {
        "w":w,
        "h":h,
        "area":a,
        "rot":page.rotation,
        "combined": is_segmented,
        "split_x": w / 2 if is_segmented else None,
        "text_dir": text_dir,
        "text": " ".join(page_text).strip(),
        "text_len": len(page_text),
        "numeric_lines": numeric_lines,
        "numeric_count": len(numeric_lines),
        "segment_lines": segment_lines,
        "segment_count": len(segment_lines),
    }    

In [ ]:
def generate_x(w, n_lines=25):
    l = w * 0.15
    r = w * 0.85

    rng = random.Random(42) #seed
    lines = sorted(
        rng.uniform(l, r)
        for _ in range(n_lines)
    )
    return lines

def consecutive_hits(hits, threshold=0):
    longest = 0
    current = 0

    for h in hits:
        if h > threshold:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest

def extract_numeric_lines(page):
    numeric_lines = []
    segment_lines = []
    page_dict = page.get_text("dict")
    margin_x = page.rect.width * 0.05
    margin_y = page.rect.height * 0.15

    work_rect = fitz.Rect(
        margin_x,
        margin_y,
        page.rect.width - margin_x,
        page.rect.height - margin_y
    )
    
    
    
    for block in page_dict["blocks"]:

        if block["type"] != 0:
            continue
        for line in block["lines"]:
            bbox = fitz.Rect(line["bbox"])
            if not work_rect.intersects(bbox):
                continue
            
            text = "".join(span["text"] for span in line["spans"]).strip()
            if NUM_LINE_RE.match(text):

                numeric_lines.append({
                    "text": text,
                    "bbox": bbox,
                    "cx": (bbox.x0 + bbox.x1) / 2
                })
                
            s_match = SECTION_RE.findall(text)
            if s_match:
                segment_lines.append(s_match)
    return numeric_lines, segment_lines

def probe_numeric_columns(page, numeric_lines, n_lines=30):

    w = page.rect.width
    probes = generate_x(w)

    hits = [0] * n_lines
    for item in numeric_lines:
        x0 = item["bbox"].x0
        x1 = item["bbox"].x1
        for i, px in enumerate(probes):

            if x0 <= px <= x1:
                hits[i] += 1

    return probes, hits

def debug_numeric_lines(page, numeric_lines, probes):

    for item in numeric_lines:

        page.draw_rect(
            item["bbox"],
            color=(1,0,0),
            width=0.8,
            overlay=True
        )

    for px in probes:

        page.draw_line(
            fitz.Point(px,0),
            fitz.Point(px,page.rect.height),
            color=(0,0,1),
            width=0.5,
            overlay=True
        )
        
def page_text_or_scanned(page):

    text = page.get_text("text").strip()
    page_rect = page.rect
    page_area = page_rect.width * page_rect.height

    if page_area <= 0:
        return "scanned"

    image_area = 0
    for img in page.get_images(full=True):
        try:
            xref = img[0]
            for rect in page.get_image_rects(xref):
                clipped = rect & page_rect
                if clipped.is_empty:
                    continue

                rect_area = clipped.width * clipped.height
                # Ignore small logos/icons
                if rect_area > page_area * 0.05:
                    image_area += rect_area

        except Exception:
            continue

    image_coverage = min(image_area / page_area, 1.0)
    blocks = page.get_text("blocks")
    text_blocks = [
        block for block in blocks if len(block) >= 5 and str(block[4]).strip()
    ]

    num_text_blocks = len(text_blocks)

    # Strong text page
    if len(text) > 100 and num_text_blocks >= 3 and image_coverage < 0.8:
        return "text"
    # Strong scanned page
    if image_coverage > 0.8 and len(text) < 100:
        return "scanned"
    # OCR scanned page
    if image_coverage > 0.9 and num_text_blocks <= 2:
        return "scanned"
    return "text" if len(text) > 100 else "scanned"

def text_dir(page, max_lines=50):
    """
    (1,0)   = NORMAL
    (0,-1)  = 90° clockwise
    (-1,0)  = 180°
    (0,1)   = 90° counter-clockwise
    """

    dirs = Counter()
    line_count = 0

    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            dirs[tuple(map(round, line["dir"]))] += 1
            line_count += 1
            if line_count >= max_lines:
                break
        if line_count >= max_lines:
            break
    if not dirs:
        return None

    p_dir = dirs.most_common(1)[0][0]
    if p_dir == (0, -1):
        return "90_CCLK"
    elif p_dir == (0, 1):
        return "90_CLK"
    elif p_dir == (-1, 0):
        return "UPSIDE_DOWN"

    return "NORMAL"
    
def detect_spread(page, ref_width, ref_height):

    width = page.rect.width
    height = page.rect.height
    w_ratio = width / ref_width
    h_ratio = height / ref_height

    is_spread = w_ratio > 1.6 and h_ratio > 0.7

    return {
        "width": width,
        "height": height,
        "width_ratio": w_ratio,
        "height_ratio": h_ratio,
        "area_ratio": (width * height) / (ref_width * ref_height),
        "spread": is_spread,
    }



In [ ]:
folder_path = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2025"

probe_data = []
pdf_data = []
files = os.listdir(folder_path)[:50]
total_files = len(files)

for idx,file in enumerate(files):
    print(f"{idx}/ {total_files}: {file}")
    pdf_path = os.path.join(folder_path, file)
    pdf_file = Path(pdf_path)
    doc = fitz.open(pdf_path)
    total_pages = doc.page_count
    start_t = time.perf_counter()
    for page_no in range(total_pages):

        page = doc[page_no]

        numeric_lines,s_lines = extract_numeric_lines(page)
        probes, hits = probe_numeric_columns(page, numeric_lines)

        row = {
            "pdf": pdf_file.stem,
            "page": page_no + 1,
            "numeric_lines": len(numeric_lines),
            # "matched_lines": "|".join(line["text"] for line in numeric_lines),
            "max_hits": max(hits),
            "total_hits":sum(hits),
            "hits_1":len([i for i in hits if i >1]),
            "cons_hit": consecutive_hits(hits, threshold=1),
            "segment_match": "YES" if len(s_lines) else "NO",
            "segment_count": len(s_lines)
            # "segment_matches":"|".join(s_lines)
            # "avg_hits": round(sum(hits) / len(hits), 2)
        }
        for i, h in enumerate(hits):
            row[f"line_{i+1}"] = h


        probe_data.append(row)
    end_t = time.perf_counter()
    
    pdf_data.append(
        {
            "pdf_name":pdf_file.stem,
            "total_pages":total_pages,
            "file_size":os.path.getsize(pdf_path),
            "time_elapsed": end_t - start_t
        }
    )
    doc.close()

In [29]:
import pandas as pd

fp = Path(folder_path)
excel_path = f"{fp.stem}_50_HITS.xlsx"

df = pd.DataFrame(probe_data)
df1 = pd.DataFrame(pdf_data)
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name ="page_wise" ,index=False)
    df1.to_excel(writer,sheet_name ="pdf_wise",index=False)

In [24]:
from collections import Counter

def top_two_modes(values):
    c = Counter(values)
    return c.most_common(2)

hits = top_two_modes(df["max_hits"])
# [(0, 82), (1, 34)]

lines = top_two_modes(df["numeric_lines"])
# [(0, 81), (1, 36)]

print(f"HITS: {hits}, LINES: {lines}")

HITS: [(1, 296), (3, 27)], LINES: [(1, 160), (2, 85)]


In [42]:
max_modes = {int(m) for m, _ in top_two_modes(df["max_hits"])}
num_modes = {int(m) for m, _ in top_two_modes(df["numeric_lines"])}

max_modes.add(0)
num_modes.add(0)

print(max_modes)
print(num_modes)

filtered = df[
    ~(
        df["max_hits"].astype(int).isin(max_modes) &
        df["numeric_lines"].astype(int).isin(num_modes)
    )
].copy()

{0, 1, 3}
{0, 1, 2}


In [44]:
filtered.to_excel("FINAL_PDF.xlsx", index = False)

In [ ]:
#get_pdf
input_pdf = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2025\2025_ADANIENT.pdf"
output_pdf = "selected_pages.pdf"

pages = [1, 3, 5, 8, 10]

src = fitz.open(input_pdf)
dst = fitz.open()

for p in pages:
    dst.insert_pdf(src, from_page=p - 1, to_page=p - 1)

dst.save(output_pdf)
dst.close()
src.close()

HIGHLIGHT DEBUG

In [23]:
def save_line_bboxes(pdf_path, output_dir):
    doc = fitz.open(pdf_path)

    for page_no in range(len(doc)):
        page = doc[page_no]
        page_dict = page.get_text("dict")

        for block in page_dict["blocks"]:
            if block["type"] != 0:
                continue

            for line in block["lines"]:
                bbox = fitz.Rect(line["bbox"])

                page.draw_rect(
                    bbox,
                    color=(1, 0, 0),      # red
                    width=0.7,
                    overlay=True
                )

    pdf_name = Path(pdf_path).stem
    out_file = Path(output_dir) / f"{pdf_name}_lines.pdf"

    doc.save(out_file)
    doc.close()

    print(out_file)

def save_block_bboxes(pdf_path, output_dir):
    doc = fitz.open(pdf_path)

    for page in doc:
        page_dict = page.get_text("dict")

        for block in page_dict["blocks"]:

            if block["type"] != 0:
                continue

            bbox = fitz.Rect(block["bbox"])

            page.draw_rect(
                bbox,
                color=(0, 0, 1),      # blue
                width=1.2,
                overlay=True
            )

    pdf_name = Path(pdf_path).stem
    out_file = Path(output_dir) / f"{pdf_name}_blocks.pdf"

    doc.save(out_file)
    doc.close()

    print(out_file)


In [30]:
#line
pdf_path = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2025\2025_ABB.pdf"
output_dir = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\eda_pdf\DRAWN"

save_line_bboxes(pdf_path, output_dir)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\eda_pdf\DRAWN\2025_ABB_lines.pdf


In [25]:
#blocks
pdf_path = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2025\2025_ELECON.pdf"
output_dir = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\eda_pdf\DRAWN"

save_block_bboxes(pdf_path, output_dir)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\eda_pdf\DRAWN\2025_ELECON_blocks.pdf
